In [1]:
# Input Image
# SET PYTHONPATH to the parent directory 
import os
import sys
import torch
import matplotlib.pyplot as plt

plt.rc('text', usetex=True)
from einops import rearrange
if os.path.abspath("..") not in sys.path:
    sys.path.append(os.path.abspath(".."))

torch.set_default_dtype(torch.float64)
MAX_ORDER = 4
FILTER_SIZE = 15

In [8]:
import numpy as np
from einops import repeat, rearrange
from hippy2d.complex_invariants_2d import get_complex_monomial
from torchvision import transforms



class TestLayer(torch.nn.Module):
    def __init__(self, 
                 max_order=4, 
                 filter_size=15,
                 gcd=False,
                 normalize_magnitude=False, 
                 masking_middles=False,
                 masking_borders=False):
        super().__init__()
        symmetric_polynomials = []
        non_symmetric_polynomials = []
        non_symmetric_exponents = []
        for p in range(0, max_order + 1):
            for q in range(0, min(max_order + 1-p, p + 1)):
                if p == q:
                    symmetric_polynomials.append((p, q))
                else:
                    non_symmetric_polynomials.append((p, q))
                    non_symmetric_exponents.append(p - q)
        non_symmetric_exponents = np.array(non_symmetric_exponents)
        coef_a = repeat(non_symmetric_exponents, 'n -> m n', m=len(non_symmetric_exponents))
        coef_c = repeat(non_symmetric_exponents, 'n -> n m', m=len(non_symmetric_exponents))

        filters = []
        types = []
        # First symmetric
        for (p, q) in symmetric_polynomials:
            filters.append(get_complex_monomial(filter_size,
                                                p, q, dtype=torch.get_default_dtype()))
            types.append(p-q)
        # Non-symmetric
        for (p, q) in non_symmetric_polynomials:
            filters.append(get_complex_monomial(filter_size,
                                                p, q, dtype=torch.get_default_dtype()))
            types.append(p-q)
        # Stack filters
        filters = torch.stack(filters, dim=0)[:, None]
        if normalize_magnitude: 
            filters /= filters.abs()

        if gcd:
            coef_gcd = np.gcd(coef_a, coef_c) 
            coef_a //= coef_gcd
            coef_c //= coef_gcd

        if masking_middles:
            for idx, type in enumerate(types):
                if type != 0: 
                    filters[idx, 0, filter_size//2, filter_size//2] = 0
                if type >= 3: 
                    filters[idx, 0, filter_size//2-1: filter_size//2+2, filter_size//2-1: filter_size//2+2] = 0
        if masking_borders:
            tukey_mask =  tukey_2d(filter_size, alpha=0.5)
            filters *= tukey_mask[None, None]

        self.register_buffer('symmetric_polynomials', torch.tensor(symmetric_polynomials))
        self.register_buffer('non_symmetric_polynomials', torch.tensor(non_symmetric_polynomials))
        self.register_buffer('filters', filters)
        self.register_buffer('coef_a', torch.tensor(coef_a, dtype=torch.get_default_dtype()))
        self.register_buffer('coef_c', torch.tensor(coef_c, dtype=torch.get_default_dtype()))
        self.register_buffer('types', torch.tensor(types, dtype=torch.uint8))

    def forward(self, x):
        # x shape (B, C, H, W)
        B, C, H, W = x.shape
        x = rearrange(x, 'b c h w -> (b c) 1 h w')
        x = torch.nn.functional.conv2d(x, self.filters, padding='same')
        symm_invariant = x[:, :len(self.symmetric_polynomials)]
        non_symmetric = x[:, len(self.symmetric_polynomials):]
        a = non_symmetric[:, :, None] ** self.coef_a[..., None, None]
        print("Non-symmetric:", non_symmetric.shape,  self.coef_a.shape)
        b = non_symmetric[:,None, :].conj() ** self.coef_c[..., None, None]
        print(b.shape)
        nonsymm_invariant = a * b
        return symm_invariant, nonsymm_invariant


from hippy2d.utils import get_testing_img, get_default_complex

def assert_equivariance(layer):  
    test_img = get_testing_img(rgb=True)
    test_img = transforms.ToTensor()(test_img).to(dtype=torch.float64)
    rotated_img = torch.rot90(test_img.clone(), 1, dims=[-2, -1])
    x = test_img[None,...].to(get_default_complex())
    rot_x = rotated_img[None,...].to(get_default_complex())
    with torch.no_grad():
        ys = layer(x)
        ys_rot = layer(rot_x)
        if not isinstance(ys, (list, tuple)):
            ys = [ys]
            ys_rot = [ys_rot]
        for y, y_rot in zip(ys, ys_rot):
            print("Going through shape:", y.shape)
            y_rot = torch.rot90(y_rot, k=-1, dims=[-2, -1])
            torch.testing.assert_close(y, y_rot)
    print("Equivariance test passed")



In [9]:
import torch 
from tqdm import tqdm
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

def get_layer_rci(layer, dataloader):
    metrics = {}
    # Save all the metrics 
    layer = layer.to(device)
    layer.eval()
    for batch in tqdm(dataloader): 
        imgs, _ = batch
        rot_imgs = torch.rot90(imgs.clone(), 1, dims=[-2, -1])
        with torch.no_grad():
            imgs = imgs.to(device)
            rot_imgs = rot_imgs.to(device)
            ys = layer(imgs)
            ys_rot = layer(rot_imgs)
            if not isinstance(ys, (list, tuple)):
                ys = [ys]
                ys_rot = [ys_rot]
            ys_rot = [torch.rot90(y_rot, k=-1, dims=[-2, -1]) for y_rot in ys_rot]
            for i, (y, y_rot) in enumerate(zip(ys, ys_rot)): 
                diff = torch.norm(y - y_rot, dim=(-2, -1)) / torch.norm(y, dim=(-2, -1))
                metrics[i] = metrics.get(i, []) + [diff.cpu()]
    return metrics 

In [10]:
# Plot all the filters 
import matplotlib.pyplot as plt
import colorsys

# Activate Latex in matplotlib
plt.rcParams['text.usetex'] = True


def complex_to_rgb(Z):
    mag = np.abs(Z)
    phase = np.angle(Z)
    norm_mag = mag / mag.max()
    
    hsv = np.zeros(Z.shape + (3,))
    hsv[..., 0] = (phase + np.pi) / (2*np.pi)   # hue
    hsv[..., 1] = 1.0                           # saturation
    hsv[..., 2] = norm_mag                      # value
    
    rgb = np.zeros_like(hsv)
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            rgb[i,j] = colorsys.hsv_to_rgb(*hsv[i,j])
    return rgb


def plot_layer_filters(layer):
    # plot all the kernels
    fig, axs = plt.subplots(1, layer.filters.shape[0], figsize=(20, 10))
    indices = layer.symmetric_polynomials.tolist() + layer.non_symmetric_polynomials.tolist()
    for i in range(layer.filters.shape[0]):
        kernel = layer.filters[i, 0].cpu().numpy()
        mean_mag = np.abs(kernel).sum()
        rgb = complex_to_rgb(kernel)
        axs[i].imshow(rgb)
        axs[i].axis('off')
        if i < len(layer.symmetric_polynomials):
            axs[i].set_title(f"(sym) $c_{{{indices[i][0]}, {indices[i][1]}}}$ - {mean_mag:.2f}", fontsize=12)
        else: 
            axs[i].set_title(f"$c_{{{indices[i][0]}, {indices[i][1]}}}$ - {mean_mag:.2f}", fontsize=12)

def print_invariance_matrix(layer):
    matrix = np.empty((layer.non_symmetric_polynomials.shape[0], layer.non_symmetric_polynomials.shape[0]),dtype=object)
    for i in range(layer.non_symmetric_polynomials.shape[0]):
        for j in range(layer.non_symmetric_polynomials.shape[0]):
            _pq = layer.non_symmetric_polynomials[i]
            _qp = (layer.non_symmetric_polynomials[j][1], layer.non_symmetric_polynomials[j][0])
            if layer.coef_a[i,j] == -1: 
                a = f"c_{{{int(_pq[0])}{int(_pq[1])}}}"
            else: 
                a = f"c_{int(_pq[0])}{int(_pq[1])}^{int(layer.coef_a[i,j])}"
            if layer.coef_c[i,j] == -1: 
                c = f"c_{{{int(_qp[0])}{int(_qp[1])}}}"
            else: 
                c = f"c_{int(_qp[0])}{int(_qp[1])}^{int(layer.coef_c[i,j])}"
            matrix[i][j] = f"{a} * {c}"
    with np.printoptions(linewidth=200, formatter={'all':lambda x: x}):
        print(matrix)


In [11]:
from tqdm import tqdm
from hippy2d.datasets import MnistRotTest

lg_dataset = MnistRotTest(data_dir="/Users/karella/Projects/rotation-invariant-neural-networks/hippy2d/data", 
                          to_complex=True, 
                          batch_size=64)
lg_dataset.prepare_data()
lg_dataset.setup("train")

In [12]:
layer = TestLayer(max_order=MAX_ORDER, filter_size=FILTER_SIZE, gcd=False)
assert_equivariance(layer)
diffs = get_layer_rci(layer, lg_dataset.train_dataloader())

Non-symmetric: torch.Size([3, 6, 256, 256]) torch.Size([6, 6])
torch.Size([3, 6, 6, 256, 256])
Non-symmetric: torch.Size([3, 6, 256, 256]) torch.Size([6, 6])
torch.Size([3, 6, 6, 256, 256])
Going through shape: torch.Size([3, 3, 256, 256])
Going through shape: torch.Size([3, 6, 6, 256, 256])
Equivariance test passed


  0%|          | 0/782 [00:00<?, ?it/s]/Users/karella/.conda/envs/hippy2d/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
  0%|          | 0/782 [00:00<?, ?it/s]


RuntimeError: Broken pipe

In [ ]:
symmetric = torch.cat(diffs[0], dim=0)
non_symmetric = torch.cat(diffs[1], dim=0)
with np.printoptions(formatter={'float_kind': '{:.3e}'.format}, linewidth=120):
    print("Max symmetric diff:", symmetric.max(dim=0)[0].numpy())
    print("Max non-symmetric diff: \n", non_symmetric.max(dim=0)[0].numpy())

In [ ]:
plot_layer_filters(layer)
print_invariance_matrix(layer)

In [ ]:
# Activate GCD
gcd_layer = TestLayer(max_order=MAX_ORDER, filter_size=FILTER_SIZE, gcd=True)
assert_equivariance(gcd_layer)
gcd_diffs = get_layer_rci(gcd_layer, lg_dataset.train_dataloader())

In [ ]:
with np.printoptions(formatter={'float_kind': '{:.3e}'.format}, linewidth=120):
    gcd_symmetric = torch.cat(gcd_diffs[0], dim=0)
    gcd_non_symmetric = torch.cat(gcd_diffs[1], dim=0)
    print("With GCD")
    print("Max symmetric diff:", gcd_symmetric.max(dim=0)[0].numpy())
    print("Max non-symmetric diff: \n", gcd_non_symmetric.max(dim=0)[0].numpy())

In [ ]:
plot_layer_filters(gcd_layer)
print_invariance_matrix(gcd_layer)

In [ ]:
layer.filters /= layer.filters.abs()

In [ ]:
layer = TestLayer(max_order=MAX_ORDER, filter_size=FILTER_SIZE, gcd=True, normalize_magnitude=True)
plot_layer_filters(layer)

In [ ]:
# Activate GCD
mag_layer = TestLayer(max_order=MAX_ORDER, filter_size=FILTER_SIZE, normalize_magnitude=True)
#assert_equivariance(mag_layer)
mag_diffs = get_layer_rci(mag_layer, lg_dataset.train_dataloader())

In [ ]:

with np.printoptions(formatter={'float_kind': '{:.1e}'.format}, linewidth=120):
    mag_symmetric = torch.cat(mag_diffs[0], dim=0)
    mag_non_symmetric = torch.cat(mag_diffs[1], dim=0)
    print("With Magnitude Normalization")
    print("Max symmetric diff:", mag_symmetric.max(dim=0)[0].numpy())
    print("Max non-symmetric diff: \n", mag_non_symmetric.max(dim=0)[0].numpy())

In [ ]:
# Activate GCD
mask_layer = TestLayer(max_order=MAX_ORDER,
                      filter_size=FILTER_SIZE,
                      gcd=True,
                      normalize_magnitude=True,
                      masking_middles=True, 
                      masking_borders=True)
assert_equivariance(mask_layer)
#mask_diffs = get_layer_rci(mask_layer, lg_dataset.train_dataloader())
mask_layer.types

In [ ]:
plot_layer_filters(mask_layer)
print_invariance_matrix(mask_layer)

In [ ]:
with np.printoptions(formatter={'float_kind': '{:.3e}'.format}, linewidth=120):
    mask_symmetric = torch.cat(mask_diffs[0], dim=0)
    mask_non_symmetric = torch.cat(mask_diffs[1], dim=0)
    print("With GCD")
    print("Max symmetric diff:", mask_symmetric.max(dim=0)[0].numpy())
    print("Max non-symmetric diff: \n", mask_non_symmetric.max(dim=0)[0].numpy())

In [ ]:
mask_layer.types

In [13]:
layer.coef_a.shape

torch.Size([6, 6])